In [8]:
%pip install --upgrade transformers
%pip install accelerate outlines langextract-outlines
%pip install langextract 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import outlines
import functools
from transformers import AutoModelForCausalLM, AutoTokenizer
from langextract_outlines import OutlinesProvider
import langextract as lx
from langextract.data import Document, ExampleData, Extraction
import os
import json
import textwrap

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

MODEL_ID = "Qwen/Qwen3.5-4B"

CORPUS_PATH = "./corpus_eu"
OUT_PATH = "./langextract_corpus_extractions_eu"


In [10]:
# Load model 

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

# Patch tokenizer to always use non-thinking mode
# Outlines calls apply_chat_template internally without enable_thinking=False,
# so we wrap it to inject the flag automatically.
_original_apply = tokenizer.apply_chat_template

@functools.wraps(_original_apply)
def _patched_apply(conversation, **kwargs):
    kwargs.setdefault("enable_thinking", False)
    return _original_apply(conversation, **kwargs)

tokenizer.apply_chat_template = _patched_apply

# Official Qwen3.5 recommended params for non-thinking / instruct mode:
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.8
model.generation_config.top_k = 20
model.generation_config.do_sample = True  # required, otherwise temp/top_p/top_k are ignored

outlines_model = outlines.from_transformers(model, tokenizer)

# Build the LangExtract provider
provider = OutlinesProvider(
    outlines_model,
    max_new_tokens=768,
)

Loading weights: 100%|██████████| 426/426 [00:00<00:00, 35502.57it/s]


In [11]:
# Load corpus documents

documents = []

for file_path in sorted(os.listdir(CORPUS_PATH), key=str.lower):
    city_name = file_path.split(".")[0]
    with open(f"{CORPUS_PATH}/{file_path}", "r", encoding="utf-8") as f:
        file_content = f.read()
    
    doc = Document(
        document_id = city_name,
        text = file_content
    )
    documents.append(doc)
print(len(documents))

135


In [12]:
# Prompt and extraction rules

CLASSES = [
    "tourist_attraction", "archaeological_site", "monument", "museum",
    "palace", "monastery", "cathedral", "square", "castle",
    "tower", "park", "lake", "theatre", "stadium", "gallery", 
    "river", "beach", "waterfall", "mountain", "market", "church",
    "synagogue", "mosque"
]

prompt = textwrap.dedent("""\
    You are an information extraction system.
    Goal: Extract structured tourism-related information about geographic entities from the text.
    Entities of interest which must be one of the following:
    """ 
    + "\n".join(f"- {clazz}" for clazz in CLASSES)+
    """
    Extract only entities related to this topic for the mentioned cities.
    Do not paraphrase or overlap entities.
    Do not invent entities that are not explicitly present in the text.
    """)

# High-quality examples to guide the model
example1 = "Rome is home to many historic landmarks. The Colosseum is one of the most famous monuments in the world. Nearby stands the Roman Forum, an important archaeological site. Tourists also visit the Vatican Museums and St. Peter's Basilica. In the historic center you can find Piazza Navona and Castel Sant'Angelo."
example2 = "Florence attracts millions of visitors every year. The Florence Cathedral dominates the skyline. Nearby is Piazza della Signoria and the historic Palazzo Vecchio. Art lovers visit the Uffizi Gallery and the Bargello Museum."
example3 = "Bruges is a medieval city in Belgium known for its canals and well-preserved architecture. The Belfry of Bruges is a prominent tower that rises above the market square. Visitors explore the Groeningemuseum for Flemish art. The Church of Our Lady contains Michelangelo's Madonna and Child. The Saint John's Hospital is one of the oldest surviving hospital buildings in Europe. Outside the city center, visitors enjoy Minnewater Park along the banks of Minnewater Lake."
example4 = "Vienna is Austria's capital and a city of imperial grandeur. The Wiener Staatsoper is one of the world's leading opera houses and a historic theatre. The Schönbrunn Palace and its gardens attract millions of visitors each year. Klosterneuburg Monastery, located just north of the city, is a major Augustinian monastery founded in the 12th century. The Prater park is home to the iconic Riesenrad ferris wheel. The Donauturm is a tall tower offering panoramic views of the Danube. Lake Neusiedl, on the eastern edge of the region, is a popular natural attraction."
example5 = "Amsterdam is known for its artistic heritage and vibrant street life. The Stedelijk Museum is a leading contemporary art gallery in the city center. The Westerkerk is a prominent Protestant church dating from the 17th century. The Sint-Nicolaasbasiliek is a major basilica near the central station. The Albert Cuyp Market is the largest outdoor market in the Netherlands, stretching along the Amstel River."
example6 = "Barcelona is a Mediterranean city with a rich cultural and natural landscape. The Camp Nou is the largest football stadium in Europe. Barceloneta Beach is the most popular urban beach in the city, stretching along the seafront. Day trips from the city often include Montserrat, a dramatic mountain with a medieval monastery at its peak. The Cascada Monumental in Ciutadella Park features a grand waterfall designed by Josep Fontserè."
example7 = "Thessaloniki is a historic port city in northern Greece. The Rotunda is an ancient monument and one of the oldest churches in the world. The White Tower is the most recognizable landmark and a tourist attraction on the city's waterfront. The Ladadika district hosts the Modiano Market, a historic covered market still active today. The Jewish Museum documents the history of the city's Sephardic community near the old Monastiriotes Synagogue. The Bey Hamam is an Ottoman-era mosque converted into an exhibition space. The Temple of Zeus Hypsistos is an ancient Macedonian temple discovered in the city center."


examples = [
    ExampleData(
        text = example1,
        extractions = [
            Extraction(extraction_class="monument",extraction_text="Colosseum"),
            Extraction(extraction_class="archaeological_site", extraction_text="Roman Forum"),
            Extraction(extraction_class="museum", extraction_text="Vatican Museums"),
            Extraction(extraction_class="cathedral", extraction_text="St. Peter's Basilica"),
            Extraction(extraction_class="square", extraction_text="Piazza Navona"),
            Extraction(extraction_class="castle", extraction_text="Castel Sant'Angelo")
        ]
    ),
    ExampleData(
        text = example2,
        extractions = [
            Extraction(extraction_class="cathedral", extraction_text="Florence Cathedral"),
            Extraction(extraction_class="square", extraction_text="Piazza della Signoria"),
            Extraction(extraction_class="palace", extraction_text="Palazzo Vecchio"),
            Extraction(extraction_class="museum", extraction_text="Uffizi Gallery"),
            Extraction(extraction_class="museum", extraction_text="Bargello Museum")
        ]
    ),
    ExampleData(
        text=example3,
        extractions=[
            Extraction(extraction_class="tower", extraction_text="Belfry of Bruges"),
            Extraction(extraction_class="museum", extraction_text="Groeningemuseum"),
            Extraction(extraction_class="cathedral", extraction_text="Church of Our Lady"),
            Extraction(extraction_class="park", extraction_text="Minnewater Park"),
            Extraction(extraction_class="lake", extraction_text="Minnewater Lake"),
        ]
    ),
    ExampleData(
        text=example4,
        extractions=[
            Extraction(extraction_class="theatre", extraction_text="Wiener Staatsoper"),
            Extraction(extraction_class="palace", extraction_text="Schönbrunn Palace"),
            Extraction(extraction_class="monastery", extraction_text="Klosterneuburg Monastery"),
            Extraction(extraction_class="park", extraction_text="Prater"),
            Extraction(extraction_class="tower", extraction_text="Donauturm"),
            Extraction(extraction_class="lake", extraction_text="Lake Neusiedl"),
        ]
    ),
    ExampleData(
        text=example5,
        extractions=[
           Extraction(extraction_class="gallery",  extraction_text="Stedelijk Museum"),
           Extraction(extraction_class="church",   extraction_text="Westerkerk"),
           Extraction(extraction_class="tourist_attraction", extraction_text="Sint-Nicolaasbasiliek"),
           Extraction(extraction_class="market",   extraction_text="Albert Cuyp Market"),
           Extraction(extraction_class="river",    extraction_text="Amstel River"),
        ]
    ),
    ExampleData(
        text=example6,
        extractions=[
            Extraction(extraction_class="stadium",   extraction_text="Camp Nou"),
            Extraction(extraction_class="beach",     extraction_text="Barceloneta Beach"),
            Extraction(extraction_class="mountain",  extraction_text="Montserrat"),
            Extraction(extraction_class="waterfall", extraction_text="Cascada Monumental"),
        ]
    ),
    ExampleData(
        text=example7,
        extractions=[
            Extraction(extraction_class="tower", extraction_text="White Tower"),
            Extraction(extraction_class="market",    extraction_text="Modiano Market"),
            Extraction(extraction_class="synagogue", extraction_text="Monastiriotes Synagogue"),
            Extraction(extraction_class="mosque",    extraction_text="Bey Hamam"),
            Extraction(extraction_class="tourist_attraction",    extraction_text="Temple of Zeus Hypsistos"),
        ]
    ),
]

In [ ]:
start_index = 0
end_index = 10

os.makedirs(OUT_PATH, exist_ok=True)

for idx, doc in enumerate(documents[start_index:end_index]):

    city_name = doc.document_id
    
    os.makedirs(f"{OUT_PATH}/{city_name}", exist_ok=True)

    try:
        result = lx.extract(
            text_or_documents=[doc],
            prompt_description=prompt,
            examples=examples,
            model=provider,
            extraction_passes=2,
            batch_length=3,
            max_workers=1,
            max_char_buffer=500,
            context_window_chars=50,
            resolver_params={"suppress_parse_errors": True}
        )[0]

        new_extractions = []
    
        for extraction in result.extractions:
            start_pos = None
            end_pos = None
    
            if extraction.char_interval:
                start_pos = extraction.char_interval.start_pos
                end_pos = extraction.char_interval.end_pos
            
            single_extraction = {
                "extraction_class": extraction.extraction_class,
                "extraction_text": extraction.extraction_text,
                "char_interval": {
                    "start_pos": start_pos,
                    "end_pos": end_pos
                },
                "alignment_status": extraction.alignment_status.value if extraction.alignment_status else None,
                "extraction_index": extraction.extraction_index,
                "group_index": extraction.group_index,
                "description": extraction.description,
                "attributes": extraction.attributes
            }
        
            new_extractions.append(single_extraction)

        dic = {
            "city": city_name,
            "extractions": new_extractions,
            "original_text": result.text
        }
    
        out_json_path = f"{OUT_PATH}/{city_name}/out.json"
        out_html_file = f"{OUT_PATH}/{city_name}/extraction_results.jsonl"
    
        with open(out_json_path, "w", encoding="utf-8") as f:
            json.dump(dic, f, indent=4)
    
        lx.io.save_annotated_documents([result], output_name="extraction_results.jsonl", output_dir=f"{OUT_PATH}/{city_name}")
    
        # Generate the visualization from the file
        html_content = lx.visualize(out_html_file)
        with open(f"{OUT_PATH}/{city_name}/visualization.html", "w", encoding="utf-8") as f:
            if hasattr(html_content, 'data'):
                f.write(html_content.data)
            else:
                f.write(html_content)
            print(f"Done: {city_name} ({len(new_extractions)} extractions)")

    except Exception as e:
        print(f"Failed: {city_name} — {e}")
        # Continue to next document instead of crashing
        continue

